# Automate Cleaning Script

In [ ]:
import os
import pandas as pd
import logging
from datetime import datetime
import warnings

INPUT_FOLDER = "dataset/"
OUTPUT_FOLDER = "data/processed/"
LOG_FOLDER = "logs/"

# Suppress specific warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Could not infer format.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*dayfirst=True.*')

# SETUP LOGGING
os.makedirs(LOG_FOLDER, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(LOG_FOLDER, "process.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


def handle_null_values(df, file_name):
    """
    Check for null values and prompt user for action if found.
    Returns the modified DataFrame based on user choice.
    """
    total_nulls = df.isnull().sum().sum()
    
    if total_nulls == 0:
        print(f"No null values found in {file_name}")
        logging.info(f"[{file_name}] No null values detected")
        return df
    
    print(f"\nNull values detected in {file_name}:")
    print(f"   Total null values: {total_nulls}")
    print("\nNull count per column:")
    null_counts = df.isnull().sum()
    for col, count in null_counts[null_counts > 0].items():
        print(f"   - {col}: {count} nulls ({count/len(df)*100:.2f}%)")
    
    while True:
        print("\nHow would you like to handle null values?")
        print("   1. Keep nulls (fill them later)")
        print("   2. Remove rows with nulls completely")
        
        choice = input("Enter your choice (1 or 2): ").strip()
        
        if choice == "1":
            print(f"Keeping null values in {file_name}")
            logging.info(f"[{file_name}] User chose to keep null values")
            return df
        
        elif choice == "2":
            initial_rows = len(df)
            df = df.dropna()
            removed_rows = initial_rows - len(df)
            print(f"Removed {removed_rows} rows containing null values from {file_name}")
            logging.info(f"[{file_name}] User chose to remove null rows. Removed {removed_rows} rows")
            return df
        
        else:
            print("Invalid choice. Please enter 1 or 2.")


def process_file(file_path):
    """
    Process a single CSV file with complete cleaning, profiling, and reporting.
    Creates organized folder structure: data/processed/{filename}/
    """
    file_name = os.path.splitext(os.path.basename(file_path))[0]

    # Create folder structure per file
    base_output_path = os.path.join(OUTPUT_FOLDER, file_name)
    reports_path = os.path.join(base_output_path, "reports")

    os.makedirs(base_output_path, exist_ok=True)
    os.makedirs(reports_path, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Processing: {file_name}")
    print(f"{'='*60}")
    logging.info(f"Processing file: {file_name}")

    #HANDLE MULTIPLE ENCODINGS
    #Common encodings in order of likelihood
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16']
    df = None
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            if encoding != 'utf-8':
                print(f"Loaded with {encoding} encoding (non-UTF-8)")
                logging.warning(f"[{file_name}] Loaded with {encoding} encoding")
            print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
            break
        except (UnicodeDecodeError, UnicodeError):
            continue
        except Exception as e:
            print(f"Error reading {file_name}: {e}")
            logging.error(f"Error reading {file_name}: {e}")
            return
    
    if df is None:
        print(f"Could not decode {file_name} with any encoding")
        logging.error(f"[{file_name}] Failed to decode with any encoding")
        return

    # CLEAN COLUMN NAMES
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("[^a-z0-9_]", "", regex=True)
    )
    logging.info(f"[{file_name}] Column names cleaned")

    # REMOVE DUPLICATES
    initial_rows = len(df)
    df = df.drop_duplicates()
    duplicates_removed = initial_rows - len(df)
    logging.info(f"[{file_name}] Removed {duplicates_removed} duplicate rows")
    print(f"Removed {duplicates_removed} duplicate rows")

    try:
        string_cols = df.select_dtypes(include=["object", "string"]).columns
    except:
    
        string_cols = df.select_dtypes(include=["object"]).columns

    for col in string_cols:
        try:
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace('', pd.NA)
        except Exception as e:
            logging.warning(f"[{file_name}] Could not process string column '{col}': {e}")
    
    logging.info(f"[{file_name}] Stripped whitespace from {len(string_cols)} string columns")

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        

        try:
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() > len(df) * 0.5:
                df[col] = converted
                logging.info(f"[{file_name}] Converted '{col}' to numeric")
                continue
        except Exception:
            pass


        try:
            converted_datetime = pd.to_datetime(df[col], errors="coerce", dayfirst=False)
            success_rate = converted_datetime.notna().sum() / len(df)
            
            if success_rate < 0.5:
                converted_datetime_alt = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
                success_rate_alt = converted_datetime_alt.notna().sum() / len(df)
                
    
                if success_rate_alt > success_rate:
                    converted_datetime = converted_datetime_alt
                    success_rate = success_rate_alt
            if success_rate > 0.5:
                df[col] = converted_datetime
                logging.info(f"[{file_name}] Converted '{col}' to datetime")
        except Exception as e:
            logging.debug(f"[{file_name}] Could not convert '{col}' to datetime: {e}")
    
    print(f"Data type conversion completed")
    
    df = handle_null_values(df, file_name)
    
    datetime_cols = df.select_dtypes(include="datetime64[ns]").columns

    for col in datetime_cols:
        try:
            df[f"{col}_year"] = df[col].dt.year
            df[f"{col}_month"] = df[col].dt.month
            df[f"{col}_month_name"] = df[col].dt.month_name()
            df[f"{col}_day"] = df[col].dt.day
            df[f"{col}_day_name"] = df[col].dt.day_name()
            df[f"{col}_quarter"] = df[col].dt.quarter
            df[f"{col}_week"] = df[col].dt.isocalendar().week
            logging.info(f"[{file_name}] Extracted date features from '{col}'")
        except Exception as e:
            logging.warning(f"[{file_name}] Could not extract features from '{col}': {e}")

    if len(datetime_cols) > 0:
        print(f"Extracted features from {len(datetime_cols)} datetime column(s)")

    # OUTLIER DETECTION
    numeric_cols = df.select_dtypes(include="number").columns
    outlier_summary = {}

    for col in numeric_cols:
        try:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = df[(df[col] < Q1 - 1.5 * IQR) | 
                          (df[col] > Q3 + 1.5 * IQR)]
            outlier_summary[col] = len(outliers)
        except Exception as e:
            logging.warning(f"[{file_name}] Could not detect outliers in '{col}': {e}")
    
    if outlier_summary:
        print(f"✓ Outlier detection completed for {len(outlier_summary)} numeric columns")
        logging.info(f"[{file_name}] Outlier detection: {outlier_summary}")

    cleaned_csv_path = os.path.join(base_output_path, "cleaned_data.csv")
    try:
        df.to_csv(cleaned_csv_path, index=False)
        print(f"✓ Saved CSV: {cleaned_csv_path}")
        logging.info(f"[{file_name}] Cleaned CSV data saved")
    except Exception as e:
        print(f"✗ Error saving CSV: {e}")
        logging.error(f"[{file_name}] Error saving CSV: {e}")
        return
    
    cleaned_parquet_path = os.path.join(base_output_path, "cleaned_data.parquet")
    try:
        df.to_parquet(cleaned_parquet_path, index=False, engine='pyarrow', compression='snappy')
        print(f"✓ Saved Parquet: {cleaned_parquet_path}")
        logging.info(f"[{file_name}] Cleaned Parquet data saved")
    except ImportError:
        # Silently skip if pyarrow not installed
        logging.debug(f"[{file_name}] Parquet save skipped (pyarrow not installed)")
    except Exception as e:
        logging.warning(f"[{file_name}] Parquet save failed: {e}")
        
    kpis = {}

    kpis["file_name"] = file_name
    kpis["total_rows"] = len(df)
    kpis["total_columns"] = len(df.columns)
    kpis["duplicates_removed"] = duplicates_removed
    kpis["total_missing_values"] = int(df.isnull().sum().sum())
    
    total_cells = len(df) * len(df.columns)
    if total_cells > 0:
        kpis["missing_percentage"] = f"{(df.isnull().sum().sum() / total_cells * 100):.2f}%"

    datetime_cols = df.select_dtypes(include="datetime64[ns]").columns
    try:
        object_cols = df.select_dtypes(include=["object", "string"]).columns
    except:
        object_cols = df.select_dtypes(include=["object"]).columns

    if len(numeric_cols) > 0:
        kpis["numeric_columns_count"] = len(numeric_cols)
        kpis["numeric_columns"] = ", ".join(numeric_cols)

    if len(datetime_cols) > 0:
        kpis["datetime_columns_count"] = len(datetime_cols)
        kpis["datetime_columns"] = ", ".join(datetime_cols)

    if len(object_cols) > 0:
        kpis["text_columns_count"] = len(object_cols)

    if outlier_summary:
        kpis["outlier_counts"] = str(outlier_summary)

    kpis["unique_values_per_column"] = str(
        {col: int(df[col].nunique()) for col in df.columns}
    )

    kpis["memory_usage_mb"] = f"{df.memory_usage(deep=True).sum() / 1024 / 1024:.2f}"
    
    kpi_df = pd.DataFrame(list(kpis.items()), columns=["Metric", "Value"])

    kpi_csv_path = os.path.join(reports_path, "kpi_summary.csv")
    kpi_excel_path = os.path.join(reports_path, "kpi_summary.xlsx")

    try:
        kpi_df.to_csv(kpi_csv_path, index=False)
        kpi_df.to_excel(kpi_excel_path, index=False, engine='openpyxl')
        print(f"✓ KPI reports generated in {reports_path}/")
        logging.info(f"[{file_name}] KPI reports generated")
    except Exception as e:
        print(f"✗ Error generating KPI reports: {e}")
        logging.error(f"[{file_name}] Error generating KPI reports: {e}")

    # GENERATE DATA SUMMARY
    summary_path = os.path.join(reports_path, "data_summary.txt")
    try:
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("=" * 60 + "\n")
            f.write(f"DATA PROCESSING SUMMARY: {file_name}\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"Processing Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Source File: {file_path}\n\n")
            
            import io
            buffer = io.StringIO()
            df.info(buf=buffer)
            f.write(buffer.getvalue())
            
            f.write("\n\n")
            f.write("BASIC STATISTICS (Numeric Columns)\n")
            f.write("=" * 60 + "\n")
            if len(numeric_cols) > 0:
                f.write(df.describe().to_string())
            else:
                f.write("No numeric columns found.\n")
            
            if outlier_summary:
                f.write("\n\n")
                f.write("OUTLIER DETECTION (IQR Method)\n")
                f.write("=" * 60 + "\n")
                for col, count in outlier_summary.items():
                    percentage = (count / len(df) * 100) if len(df) > 0 else 0
                    f.write(f"{col}: {count} outliers ({percentage:.2f}%)\n")
            
            f.write("\n\n")
            f.write("DATA PROFILING (Unique Value Counts)\n")
            f.write("=" * 60 + "\n")
            for col in df.columns:
                unique_count = df[col].nunique()
                total_count = len(df)
                percentage = (unique_count / total_count * 100) if total_count > 0 else 0
                f.write(f"{col}: {unique_count} unique values ({percentage:.2f}% of total)\n")
                
            f.write("\n\n")
            f.write("MISSING VALUES SUMMARY\n")
            f.write("=" * 60 + "\n")
            missing_summary = df.isnull().sum()
            missing_summary = missing_summary[missing_summary > 0]
            if len(missing_summary) > 0:
                f.write(missing_summary.to_string())
            else:
                f.write("No missing values found.\n")
        
        print(f"Data summary saved: {summary_path}")
        logging.info(f"[{file_name}] Data summary generated")
    except Exception as e:
        logging.error(f"[{file_name}] Error generating data summary: {e}")

    print(f"Completed: {file_name}\n")
    logging.info(f"[{file_name}] Processing completed successfully")


def main():
    """
    Main function to process all CSV files in INPUT_FOLDER.
    Creates organized folder structure for each dataset.
    """
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Find all CSV files
    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".csv")]

    if not files:
        print("No CSV files found in data/raw/")
        logging.error("No CSV files found in data/raw/")
        return

    print(f"\nFound {len(files)} CSV file(s) to process:")
    for i, file in enumerate(files, 1):
        print(f"   {i}. {file}")
    print()

    successful = 0
    failed = 0
    
    for file in files:
        file_path = os.path.join(INPUT_FOLDER, file)
        try:
            process_file(file_path)
            successful += 1
        except Exception as e:
            print(f"Fatal error processing {file}: {e}")
            logging.error(f"Fatal error processing {file}: {e}")
            failed += 1
            continue

    print("\n" + "="*60)
    print("PROCESSING COMPLETE!")
    print("="*60)
    print(f"Successfully processed: {successful} file(s)")
    if failed > 0:
        print(f"Failed: {failed} file(s) (check logs for details)")
    print(f"Output location: {OUTPUT_FOLDER}")
    print(f"Logs saved to: {LOG_FOLDER}")
    
    if successful > 0:
        print("\nFolder structure created:")
        print("data/processed/")
        for file in files:
            file_name = os.path.splitext(file)[0]
            # Only show structure for successfully processed files
            output_path = os.path.join(OUTPUT_FOLDER, file_name)
            if os.path.exists(output_path):
                print(f"  ├── {file_name}/")
                print(f"  │   ├── cleaned_data.csv")
                if os.path.exists(os.path.join(output_path, "cleaned_data.parquet")):
                    print(f"  │   ├── cleaned_data.parquet")
                print(f"  │   └── reports/")
                print(f"  │       ├── kpi_summary.csv")
                print(f"  │       ├── kpi_summary.xlsx")
                print(f"  │       └── data_summary.txt")
    
    logging.info(f"Processing complete: {successful} successful, {failed} failed")

if __name__ == "__main__":
    main()